# RI #1: Retrieval Evaluation and the Baseline RAG Pipeline

This notebook introduces the retrieval interface and evaluation harness used throughout the reference implementations. It establishes a reproducible baseline on contract questions before adding advanced retrieval or agent techniques.

By the end, you should be able to:

- Load a pinned corpus and score BM25, dense retrieval, and a shuffled negative control.
- Interpret retrieval failure and mean reciprocal rank (MRR), and distinguish retrieval quality from answer correctness.
- Evaluate abstention on unanswerable questions and recognize the effect of chunk size on comparisons.
- Trace a single-pass answering pipeline and reuse the harness with another retriever.

## 1. Evaluation Data and Saved Measurements

Run the cells in order. Keep `RUN_EXPENSIVE=False` for the first run. Corpus loaders download and verify pinned data on first use; BM25 and the controls run locally. Dense retrieval reports and generation examples are read from saved measurements through the shared result cache. This default path needs no API key and does not load an embedding model. Enabling the flag recomputes dense retrieval and makes four generation calls; the final section explains the model and credentials.

The first evaluation slice contains **20 CUAD test contracts**, split into **954 chunks** of 1,200 characters with 200 characters of overlap. Its 820 questions include **250 answerable** and **570 unanswerable** questions. Relevance judgments (`corpus.qrels`) are derived from the annotated answer spans. Retrieval is scoped to each question's own contract so all baseline arms use the same candidate pool. See [CORPUS.md](CORPUS.md) for versions, digests, and attribution.

In [1]:
import hashlib
from importlib.metadata import version
from pathlib import Path

from aieng.advanced_rag.cache import CacheKey, TieredResultCache
from aieng.advanced_rag.corpora import contract_nli, cuad
from aieng.advanced_rag.evaluation.metrics import (
    budget_k,
    retrieved_chars,
    score_null_detection,
    score_retrieval,
)
from aieng.advanced_rag.metering import UsageMeter
from aieng.advanced_rag.pipeline.answering import SingleShotPolicy
from aieng.advanced_rag.pipeline.grading import HeuristicGrader
from aieng.advanced_rag.pipeline.spine import AdvancedRagSpine, SpineConfig
from aieng.advanced_rag.retrieval.retrievers import ShuffledRetriever, run_retriever
from aieng.advanced_rag.retrieval.sparse import BM25Retriever


# Embedding and generation ship off. Enable the flag to refresh the measured
# artifacts; the default path reads them through the shared result cache.
RUN_EXPENSIVE = False

K = 10
DENSE_MODEL = "BAAI/bge-small-en-v1.5"

# This fetch is intentionally not behind the flag. It downloads the immutable
# CUAD test split on first use, verifies both archive and member SHA-256 digests,
# and reuses the verified cache thereafter.
cuad_path = cuad.fetch_cuad_test()
corpus = cuad.build_slice(cuad_path, max_documents=20)
summary = corpus.summary()
assert summary == {
    "contracts": 20,
    "chunks": 954,
    "queries": 820,
    "answerable_queries": 250,
    "null_queries": 570,
    "null_rate": 0.6951,
    "mean_gold_chunks": 2.09,
}
print(
    f"{summary['contracts']} contracts, {summary['chunks']} chunks, "
    f"{summary['queries']} questions ({summary['answerable_queries']} answerable, "
    f"{summary['null_queries']} null; null rate {summary['null_rate']:.4f})"
)


REPO = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "aieng-advanced-rag").is_dir())
RI = REPO / "implementations" / "regulatory_policy_qa"
cache = TieredResultCache(global_dir=RI / "outputs" / "baseline_cache")


def measured_result(key, compute):
    """Read a measured artifact or explicitly refresh it."""
    if RUN_EXPENSIVE:
        return cache.load_or_compute(key, compute, force_refresh=True).value
    cached = cache.get(key)
    if cached is None:
        raise FileNotFoundError("Missing measured artifact; enable RUN_EXPENSIVE to create it")
    return cached.value

20 contracts, 954 chunks, 820 questions (250 answerable, 570 null; null rate 0.6951)


## 2. Lexical Retrieval and a Negative Control

Each retriever follows the same workflow: **index the chunks**, **retrieve for each question**, then **score the ranked results against the relevance judgments**. `run_retriever` applies the retriever to the query set; `score_retrieval` computes the metrics at `k=10`.

| Metric | Definition | Interpretation |
| --- | --- | --- |
| Retrieval failure | Fraction of answerable questions with no gold chunk in the top `k` results | Lower is better; it identifies questions whose retrieved context lacks an annotated answer-bearing chunk. |
| Mean reciprocal rank (MRR) | Mean of `1 / rank` for the first gold chunk, with zero when none is retrieved | Higher is better; a gold chunk at rank 1 contributes 1, while one at rank 4 contributes 0.25. |

These metrics use only the **250 answerable questions**. The 570 unanswerable questions have no gold passage and are evaluated separately for abstention.

**BM25** is the lexical baseline and requires no embedding model or API key. **Shuffled retrieval** is a negative control: it returns the same contract's chunks in a fixed random order, with no ranking signal. Beating this control shows the value of ranking within this candidate pool; it does not establish that generated answers are correct.

In [2]:
bm25 = BM25Retriever()
bm25.index(corpus.chunks)
bm25_runs = run_retriever(bm25, corpus.answerable, k=K)
bm25_report = score_retrieval(bm25_runs, corpus.qrels, k=K)

control = ShuffledRetriever(seed=0)
control.index(corpus.chunks)
control_runs = run_retriever(control, corpus.answerable, k=K)
control_report = score_retrieval(control_runs, corpus.qrels, k=K)

assert (bm25_report.retrieval_failure_rate, bm25_report.mrr) == (0.2600, 0.4528)
assert (control_report.retrieval_failure_rate, control_report.mrr) == (0.4800, 0.1895)
print(f"BM25      failure {bm25_report.retrieval_failure_rate:.4f}  MRR {bm25_report.mrr:.4f}")
print(f"control   failure {control_report.retrieval_failure_rate:.4f}  MRR {control_report.mrr:.4f}")

BM25      failure 0.2600  MRR 0.4528
control   failure 0.4800  MRR 0.1895


## 3. Dense Retrieval: Reproducing and Comparing Baselines

Both dense arms use the local `BAAI/bge-small-en-v1.5` embedding model, the same CUAD slice, contract scope, and retrieval depth. The comparison changes only the query prompt.

| Dense configuration | Retrieval failure | MRR |
| --- | ---: | ---: |
| Historical baseline, `query_prompt=""` | 0.2400 | 0.4348 |
| Shipped BGE query-prompt default | 0.2400 | 0.4632 |

The promptless configuration reproduces the required baseline. Applying BGE's published query prompt improves MRR on this slice without changing the failure rate: relevant chunks move higher in the ranking, while the fraction of questions with no gold chunk remains the same.

With `RUN_EXPENSIVE=False`, the cell reads measured reports through `aieng.advanced_rag.cache` and labels them as saved measurements. With the flag enabled, it embeds 954 chunks for each arm on the CPU, recomputes the reports, and checks the expected figures to four decimal places. No hosted embedding API is used. Cache identities include the corpus digest and retrieval settings; changing those settings requires a new measurement.

Read the dense results alongside BM25 and the control above. Dense retrieval fails on 0.24 of the 250 answerable questions against BM25's 0.26, a gap of 5 questions with no interval reported, while the promptless dense arm has a lower MRR. On this slice the two retrievers differ more in where they rank the first relevant passage than in whether they find one, so report both metrics.

In [3]:
if RUN_EXPENSIVE:
    from aieng.advanced_rag.retrieval.embedders import SentenceTransformerEmbedder
    from aieng.advanced_rag.retrieval.retrievers import DenseRetriever


DENSE_PROMPTLESS = (0.2400, 0.4348)
DENSE_WITH_PROMPT = (0.2400, 0.4632)


def dense_key(query_prompt):
    """Name every input that determines the dense retrieval measurement."""
    return CacheKey(
        "ri1",
        "retrieval-report",
        "bge-small-en-v1.5",
        {
            "corpus_sha256": hashlib.sha256(cuad_path.read_bytes()).hexdigest(),
            "max_documents": 20,
            "chunk_chars": 1200,
            "overlap_chars": 200,
            "model": DENSE_MODEL,
            "query_prompt": query_prompt,
            "scope": "contract",
            "k": K,
        },
    )


def dense_measurement(report, embedder):
    """Record measured rates, embedding revision, and software versions."""
    return {
        "failure": report.retrieval_failure_rate,
        "mrr": report.mrr,
        "model_revision": embedder._model._first_module().auto_model.config._commit_hash,
        "versions": {name: version(name) for name in ("sentence-transformers", "transformers", "torch", "numpy")},
    }


def measure_promptless():
    """Recompute the historical dense baseline without a query prompt."""
    embedder = SentenceTransformerEmbedder(DENSE_MODEL, query_prompt="")
    dense_promptless = DenseRetriever(embedder)
    dense_promptless.index(corpus.chunks)
    report = score_retrieval(run_retriever(dense_promptless, corpus.answerable, k=K), corpus.qrels, k=K)
    return dense_measurement(report, embedder)


def measure_prompted():
    """Recompute the dense arm with the published BGE query prompt."""
    embedder = SentenceTransformerEmbedder(DENSE_MODEL)
    dense_prompted = DenseRetriever(embedder)
    dense_prompted.index(corpus.chunks)
    report = score_retrieval(run_retriever(dense_prompted, corpus.answerable, k=K), corpus.qrels, k=K)
    return dense_measurement(report, embedder)


promptless = measured_result(dense_key(""), measure_promptless)
prompted = measured_result(dense_key("Represent this sentence for searching relevant passages: "), measure_prompted)
dense_promptless_result = (promptless["failure"], promptless["mrr"])
dense_prompted_result = (prompted["failure"], prompted["mrr"])
source = "recomputed" if RUN_EXPENSIVE else "committed measurement"

assert tuple(round(value, 4) for value in dense_promptless_result) == DENSE_PROMPTLESS
assert tuple(round(value, 4) for value in dense_prompted_result) == DENSE_WITH_PROMPT
print(f"dense, promptless   failure {dense_promptless_result[0]:.4f}  MRR {dense_promptless_result[1]:.4f}  ({source})")
print(f"dense, BGE prompt   failure {dense_prompted_result[0]:.4f}  MRR {dense_prompted_result[1]:.4f}  ({source})")

dense, promptless   failure 0.2400  MRR 0.4348  (committed measurement)
dense, BGE prompt   failure 0.2400  MRR 0.4632  (committed measurement)


## 4. Abstention Evaluation on Unanswerable Questions

An unanswerable, or **null**, question has no annotated answer in its contract. Returning passages does not establish answerability: BM25 and dense retrieval return their best matches even when none answers the question. Null detection therefore evaluates the downstream **answer-or-abstain decision** over all **820 questions**.

- **Abstention precision:** among questions the system refuses, the fraction that are actually null.
- **Abstention recall:** among null questions, the fraction on which the system abstains.
- **False-answer rate:** among null questions, the fraction on which the system attempts an answer.

The cell scores two reference policies rather than a learned detector. **Always answering** gives a false-answer rate of 1.0. **Always abstaining** gives recall of 1.0, but precision is only the corpus null rate (0.6951), and it refuses all 250 answerable questions. The harness reports precision as zero when there are no abstentions.

These controls illustrate why abstention must be evaluated alongside correctness on answerable questions: refusing everything prevents false answers but does not produce a useful QA system. Gold null labels are used for evaluation only, not to make a system's answerability decision.

In [4]:
null_ids = {query.query_id for query in corpus.null_queries}
always_answer = score_null_detection({query.query_id: False for query in corpus.queries}, null_ids)
always_abstain = score_null_detection({query.query_id: True for query in corpus.queries}, null_ids)

assert always_answer.to_dict() == {
    "null_queries": 570,
    "answerable_queries": 250,
    "abstain_precision": 0.0,
    "abstain_recall": 0.0,
    "false_answer_rate": 1.0,
}
assert always_abstain.to_dict() == {
    "null_queries": 570,
    "answerable_queries": 250,
    "abstain_precision": 0.6951,
    "abstain_recall": 1.0,
    "false_answer_rate": 0.0,
}
print(f"{'policy':<19}{'precision':>12}{'recall':>10}{'false-answer':>14}")
for label, report in (("always answer", always_answer), ("always abstain", always_abstain)):
    print(
        f"{label:<19}{report.abstain_precision:>12.4f}{report.abstain_recall:>10.4f}{report.false_answer_rate:>14.4f}"
    )

policy                precision    recall  false-answer
always answer            0.0000    0.0000        1.0000
always abstain           0.6951    1.0000        0.0000


## 5. Chunk Size and the Retrieved-Text Budget

Holding `k` fixed does not hold the amount of retrieved text fixed. Ten 2,400-character chunks can supply roughly four times as much text as ten 600-character chunks. A lower retrieval-failure rate may therefore reflect a larger context allowance as well as a change in ranking quality.

The comparison holds BM25, the contracts, and the questions fixed, and evaluates chunk sizes of **600**, **1,200**, and **2,400 characters**, each with one-sixth overlap. It reports two operating points:

1. **Fixed depth:** retrieve the top ten chunks from each configuration.
2. **Approximately equal text budget:** use `budget_k` to choose a depth that reaches a mean budget of approximately 10,000 characters per question, within the search depth of 20.

The output includes the measured character counts so any remaining budget difference is visible. Whole chunks make the budgets approximate; retrieved characters can include overlapping text and are not equivalent to unique contract coverage or model tokens. BM25 keeps this diagnostic fast and independent of embedding-model calls.

In [5]:
CHAR_BUDGET = 10_000
MAX_DEPTH = 20
chunk_rows = []

for chunk_chars, overlap_chars in ((600, 100), (1200, 200), (2400, 400)):
    sliced = cuad.build_slice(
        cuad_path,
        max_documents=20,
        chunk_chars=chunk_chars,
        overlap_chars=overlap_chars,
    )
    retriever = BM25Retriever()
    retriever.index(sliced.chunks)
    deep_runs = run_retriever(retriever, sliced.answerable, k=MAX_DEPTH)
    chars_by_id = {chunk.chunk_id: len(chunk.text) for chunk in sliced.chunks}

    fixed_runs = {query_id: ranking[:K] for query_id, ranking in deep_runs.items()}
    fixed_report = score_retrieval(fixed_runs, sliced.qrels, k=K)
    fixed_chars = round(retrieved_chars(fixed_runs, chars_by_id, K))

    fair_k = budget_k(deep_runs, chars_by_id, CHAR_BUDGET, depth=MAX_DEPTH)
    fair_runs = {query_id: ranking[:fair_k] for query_id, ranking in deep_runs.items()}
    fair_report = score_retrieval(fair_runs, sliced.qrels, k=fair_k)
    fair_chars = round(retrieved_chars(fair_runs, chars_by_id, fair_k))
    chunk_rows.append(
        (
            chunk_chars,
            len(sliced.chunks),
            fixed_chars,
            fixed_report.retrieval_failure_rate,
            fair_k,
            fair_chars,
            fair_report.retrieval_failure_rate,
        )
    )

assert chunk_rows == [
    (600, 1902, 5867, 0.3040, 18, 10387, 0.2480),
    (1200, 954, 11460, 0.2600, 9, 10328, 0.2720),
    (2400, 480, 21462, 0.1520, 5, 11310, 0.2840),
]
print("chunk  chunks | fixed k chars  failure | budget k chars  failure")
for chunk_chars, chunks, fixed_chars, fixed_failure, fair_k, fair_chars, fair_failure in chunk_rows:
    print(
        f"{chunk_chars:>5}{chunks:>8} | {K:>7}{fixed_chars:>6}{fixed_failure:>9.4f} | "
        f"{fair_k:>6}{fair_chars:>6}{fair_failure:>9.4f}"
    )

chunk  chunks | fixed k chars  failure | budget k chars  failure
  600    1902 |      10  5867   0.3040 |     18 10387   0.2480
 1200     954 |      10 11460   0.2600 |      9 10328   0.2720
 2400     480 |      10 21462   0.1520 |      5 11310   0.2840


## 6. Interpreting the Chunk-Size Comparison

At fixed `k=10`, the 2,400-character configuration has a failure rate of **0.1520**, compared with **0.3040** for 600-character chunks. It also retrieves about **21,462 characters**, compared with **5,867**. This comparison changes both the chunking and the amount of text available downstream.

At approximately equal character budgets, the ordering reverses: failure is **0.2480** for 600-character chunks and **0.2840** for 2,400-character chunks. The larger chunks no longer have the best retrieval-failure rate.

The practical lesson is to report retrieval quality together with the text budget. This experiment demonstrates a confound on this corpus; it does not establish a universally optimal chunk size. A downstream answering experiment must also account for its own passage limits and truncation.

## 7. From Retrieval to a Single-Pass Answering Pipeline

ContractNLI provides a second contract dataset with labelled `NotMentioned` questions. Its first 20 test contracts contain **340 questions: 179 answerable and 161 null**. Both corpus loaders work with the expensive flag off and verify their pinned data.

The answering baseline connects scoped BM25 retrieval to the shared `AdvancedRagSpine`. It uses a model-free `HeuristicGrader`, `SingleShotPolicy`, and `max_attempts=1`. The grade is advisory; the generator decides whether the retrieved sources support an answer. No corrective rewriter is configured. See [PIPELINE.md](PIPELINE.md) for the policy and abstention rules.

Each corpus contributes its first answerable and first null question, giving **four execution examples**. With `RUN_EXPENSIVE=False`, their measured outcomes are replayed from the shared cache without API calls. With the flag enabled, the example makes four generation calls through the Vector proxy using the shared `choose_models()` settings, with reasoning disabled. Credentials use the shared resolver; no credentials are needed for replay. The committed measurement used `Qwen3_6-35B-A3B`. Zero temperature removes sampling variance but does not guarantee identical answers across serving backends.

The table shows the gold answerability label, the pipeline's abstention decision, resolved citation count, number of attempts, and the original measured **cold-call count**. That last column records the work that produced the saved artifact, not new calls made during replay. Contract passages and generated answer text are not printed.

These examples demonstrate pipeline wiring, not an accuracy estimate. In the saved CUAD null example, the system attempts an answer despite resolving a citation. A citation alone does not establish correctness or answerability. A full evaluation must judge answer correctness separately and report abstention over the complete query set.

In [6]:
contract_nli_path = contract_nli.fetch_contract_nli()
nda_corpus = contract_nli.build_slice(contract_nli_path, max_documents=20)
nda_summary = nda_corpus.summary()
assert (
    nda_summary["contracts"],
    nda_summary["queries"],
    nda_summary["answerable_queries"],
    nda_summary["null_queries"],
) == (20, 340, 179, 161)
print("ContractNLI: 20 contracts, 340 questions (179 answerable, 161 null)")

ContractNLI: 20 contracts, 340 questions (179 answerable, 161 null)


In [7]:
GENERATION_MODEL = "Qwen3_6-35B-A3B"
if RUN_EXPENSIVE:
    from aieng.advanced_rag.cache.paths import root as cache_root
    from aieng.advanced_rag.graphrag.proxy import RateLimiter, ResponseCache, govern
    from aieng.advanced_rag.models import choose_models
    from aieng.advanced_rag.pipeline.generate import ChatCompleter
    from aieng.advanced_rag.proxy import resolve_api_key, resolve_base_url
    from openai import OpenAI

    MODELS = choose_models(show=True)
    GENERATION_MODEL = MODELS.generation.model
    complete = ChatCompleter(
        govern(
            OpenAI(
                base_url=resolve_base_url(start=REPO), api_key=resolve_api_key(start=REPO), timeout=120, max_retries=0
            ),
            cache=ResponseCache(cache_root() / "responses/ri1-baseline.sqlite"),
            limiter=RateLimiter(per_minute=30),
        ),
        GENERATION_MODEL,
    )

pipeline_rows = []
for label, prepared, data_path in (("CUAD", corpus, cuad_path), ("ContractNLI", nda_corpus, contract_nli_path)):
    lexical = BM25Retriever()
    lexical.index(prepared.chunks)
    for query in (prepared.answerable[0], prepared.null_queries[0]):
        result_key = CacheKey(
            "ri1",
            "pipeline-example",
            GENERATION_MODEL,
            {
                "corpus_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
                "query_id": query.query_id,
                "max_documents": 20,
                "chunk_chars": 1200,
                "overlap_chars": 200,
                "retriever": "bm25",
                "scope": "contract",
                "top_k": 10,
                "evidence_budget": 5,
                "max_attempts": 1,
                "require_citations": True,
                "policy": "SingleShotPolicy",
                "grader": "HeuristicGrader",
                "temperature": 0.0,
                "thinking": False,
                "max_tokens": 512,
            },
        )

        def run_single_pass(query=query, lexical=lexical, label=label):
            """Run one scoped retrieval and one grounded generation call."""
            meter = UsageMeter()
            spine = AdvancedRagSpine(
                retrieve=lexical,
                grader=HeuristicGrader(),
                complete=complete,
                config=SpineConfig(max_attempts=1),
                policy=SingleShotPolicy(),
            )
            with meter.question(query.query_id):
                result = spine.answer(query.query_id, query.question, query.doc_id)
            assert len(result.attempts) == 1
            assert meter.totals().calls == 1
            assert meter.totals().empty_responses == 0
            return {
                "corpus": label,
                "query_id": query.query_id,
                "abstained": result.abstained,
                "citations": len(result.answer.citations),
                "attempts": len(result.attempts),
                "usage": meter.totals().to_dict(),
            }

        pipeline_rows.append({**measured_result(result_key, run_single_pass), "gold_null": not query.is_answerable})

print(f"single-pass model: {GENERATION_MODEL}; scoped BM25; reasoning disabled")
print("corpus       query/category         gold null  abstained  citations  attempts  cold calls")
for row in pipeline_rows:
    assert row["attempts"] == 1 and row["usage"]["calls"] == 1
    display_id = row["query_id"].rsplit("__", 1)[-1][:21]
    print(
        f"{row['corpus']:<13}{display_id:<23}{str(row['gold_null']):<11}{str(row['abstained']):<11}"
        f"{row['citations']:<11}{row['attempts']:<10}{row['usage']['calls']}"
    )

single-pass model: Qwen3_6-35B-A3B; scoped BM25; reasoning disabled
corpus       query/category         gold null  abstained  citations  attempts  cold calls
CUAD         Document Name          False      True       0          1         1
CUAD         Agreement Date         True       False      1          1         1
ContractNLI  1::nda-16              False      False      3          1         1
ContractNLI  1::nda-11              True       True       0          1         1


## 8. Reusing the Harness in Your Own Experiments

To evaluate another retriever, replace the BM25 instance and retain the same indexing, retrieval, and scoring workflow. A retriever implementing the shared interface can be scored without changing the harness. Use `aieng.advanced_rag.retrieval.conformance.assert_conforms` to check a custom implementation before running a comparison.

For a useful comparison:

- Keep the corpus slice, query set, relevance judgments, contract scope, and retrieval depth fixed when comparing ranking methods. Include the shuffled control.
- When comparing chunkings, report both fixed-depth results and results at a controlled text budget, including the actual retrieved character counts.
- Score retrieval on answerable questions and abstention over the complete question set. Evaluate generated-answer correctness separately.
- Identify whether a result was recomputed or replayed, and record the configuration and model usage that produced it.

This notebook establishes the baseline that later retrieval and agent exercises can reuse. The corrective loop remains an optional measured experiment rather than part of the recommended single-pass path. For setup and repository navigation, continue with [GUIDE.md](GUIDE.md). For the cached null-arm comparison and its separate correctness and grounding judgments, see [GRAPHRAG_NULL_ARM.md](GRAPHRAG_NULL_ARM.md).